## Import Dependencies

In [1]:
from transformers import AutoTokenizer
import polars as pl
from datasets import Dataset

## Import Data

In [2]:
train_path = "/content/complete_train.parquet"
test_path = "/content/complete_test.parquet"

In [3]:
train_df = pl.scan_parquet(train_path)
test_df = pl.scan_parquet(test_path)

In [4]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length
str,str,str,i64,str,str,str,str,i64
"""3VKW""","""A""","""SER""",1,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""TYR""",2,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""THR""",3,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""ARG""",4,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
"""3VKW""","""A""","""SER""",5,"""A""",""" ""","""3VKW_1""","""SYTRSEEIESLEQFHMATASSLIHKQMCSI…",446
…,…,…,…,…,…,…,…,…
"""5XVT""","""A""","""LEU""",693,"""A""","""T""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""ARG""",694,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697
"""5XVT""","""A""","""SER""",695,"""A""",""".""","""5XVT_1""","""MGSSHHHHHHSSGLVPRGSHMSSVDQKAIS…",697


## Process Data

### Add Labels

In [5]:
train_df = (train_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))

test_df = (test_df
            .with_columns(label = pl.col('secondary_structure').rank(method='dense') - 1)
              .cast({'label': pl.Int64}))


### Removing Duplicates on Index
Some indices have two amino acids, so remove the one without a secondary structure (label = 0)

In [6]:
train_removed = (
    train_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'label', 'secondary_structure'])
)

train_df = (
    train_df.join(train_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label')),
                          secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure'))
            )
            .drop(['amino_acid_right', 'label_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [7]:
test_removed = (
    test_df.sort(['id', 'asym_id', 'index', 'label'])
            .group_by(['id', 'asym_id', 'index'])
            .agg(['amino_acid', 'label', 'secondary_structure'])
            .filter((pl.col('amino_acid').list.len() > 1))
            .with_columns(
                amino_acid = pl.col('amino_acid').list.slice(1),
                label = pl.col('label').list.slice(1),
                secondary_structure = pl.col('secondary_structure').list.slice(1),
            )
            .explode(['amino_acid', 'label', 'secondary_structure'])
)

test_df = (
    test_df.join(test_removed, on=['id', 'asym_id', 'index'], how='left')
            .with_columns(
                          amino_acid = pl.when(~(pl.col('amino_acid_right').is_null()))
                                        .then(pl.col('amino_acid_right'))
                                        .otherwise(pl.col('amino_acid')),
                         label = pl.when(~(pl.col('label_right').is_null()))
                                        .then(pl.col('label_right'))
                                        .otherwise(pl.col('label')),
                          secondary_structure = pl.when(~(pl.col('secondary_structure_right').is_null()))
                                        .then(pl.col('secondary_structure_right'))
                                        .otherwise(pl.col('secondary_structure'))
            )
            .drop(['amino_acid_right', 'label_right', 'secondary_structure_right'])
            .unique(['id', 'asym_id', 'index'])
            .sort(['id', 'asym_id', 'index'])
)


In [8]:
test_df.collect()

id,asym_id,amino_acid,index,strand_id,secondary_structure,rcsb_id,sequence,length,label
str,str,str,i64,str,str,str,str,i64,i64
"""1A1X""","""A""","""GLY""",1,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""SER""",2,"""A""",""" ""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,0
"""1A1X""","""A""","""ALA""",3,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLY""",4,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
"""1A1X""","""A""","""GLU""",5,"""A""",""".""","""1A1X_1""","""GSAGEDVGAPPDHLWVHQEGIYRDEYQRTW…",108,1
…,…,…,…,…,…,…,…,…,…
"""7ODC""","""A""","""ILE""",420,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""GLN""",421,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0
"""7ODC""","""A""","""SER""",422,"""A""",""" ""","""7ODC_1""","""MSSFTKDEFDCHILDEGFTAKDILDQKINE…",424,0


#### Remove Duplicates on Asym ID
Some sequences are exactly the same and have the same secondary structures, so choose only one Asym ID (first ID)

In [9]:
train_df = (

  (train_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['secondary_structure', 'label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'secondary_structure', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['secondary_structure', 'label', 'index'])

)

test_df = (

  (test_df.sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
                .agg(['secondary_structure', 'label', 'index'])

  ).sort('asym_id')
    .group_by(['id', 'sequence', 'secondary_structure', 'label', 'index'])
      .agg(['asym_id'])
        .with_columns(asym_id = pl.col('asym_id').list.slice(0,1))
          .explode(['asym_id'])
            .explode(['secondary_structure', 'label', 'index'])

)



In [10]:
test_df.collect()

id,sequence,secondary_structure,label,index,asym_id
str,str,str,i64,i64,str
"""5J1G""","""GSHMQEESRCQRCISELKDIRLQLEACETR…",""" """,0,1,"""B"""
"""5J1G""","""GSHMQEESRCQRCISELKDIRLQLEACETR…",""" """,0,2,"""B"""
"""5J1G""","""GSHMQEESRCQRCISELKDIRLQLEACETR…",""" """,0,3,"""B"""
"""5J1G""","""GSHMQEESRCQRCISELKDIRLQLEACETR…",""" """,0,4,"""B"""
"""5J1G""","""GSHMQEESRCQRCISELKDIRLQLEACETR…",""" """,0,5,"""B"""
…,…,…,…,…,…
"""4P3A""","""GANLHLLRQKIEEQAAKYKHSVPKKCCYDG…",""" """,0,75,"""B"""
"""4P3A""","""GANLHLLRQKIEEQAAKYKHSVPKKCCYDG…",""" """,0,76,"""B"""
"""4P3A""","""GANLHLLRQKIEEQAAKYKHSVPKKCCYDG…",""" """,0,77,"""B"""


### Prepare for Tokenization

In [11]:
train_agg = (train_df
              .select(['id', 'asym_id', 'sequence', 'index', 'secondary_structure','label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['secondary_structure','index', 'label'])
            )

test_agg = (test_df
              .select(['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'])
              .sort(['id', 'asym_id', 'index'])
              .group_by(['id', 'asym_id', 'sequence'])
              .agg(['secondary_structure','index', 'label'])
            )

In [12]:
train_ds = Dataset.from_polars(train_agg.collect())
test_ds = Dataset.from_polars(test_agg.collect())

### Tokenization

In [13]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

def tokenize_and_label(examples):
  tokenized_inputs = tokenizer(examples["sequence"],
                                return_tensors="pt",
                                add_special_tokens=False,
                                padding=False)
  tokenized_inputs["input_ids"] = tokenized_inputs["input_ids"][0]
  tokenized_inputs["attention_mask"] = tokenized_inputs["attention_mask"][0]
  return tokenized_inputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [14]:
tokenized_train = train_ds.map(tokenize_and_label, batched=False)
tokenized_test = test_ds.map(tokenize_and_label, batched=False)

Map:   0%|          | 0/11425 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

## Output Data

### Dataset to DataFrame

In [15]:
output_train = Dataset.to_polars(tokenized_train).lazy()
output_test = Dataset.to_polars(tokenized_test).lazy()

In [16]:
output_train = (output_train
                .join(train_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'],
                      how='inner')
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask'])
                )

output_test = (output_test
                .join(test_agg,
                      on=['id', 'asym_id', 'sequence', 'index', 'secondary_structure', 'label'],
                      how='inner')
                .explode(['secondary_structure', 'index', 'label', 'input_ids', 'attention_mask'])
                )


In [17]:
output_test.collect()

id,asym_id,sequence,secondary_structure,index,label,input_ids,attention_mask
str,str,str,str,i64,i64,i32,i8
"""2QR6""","""A""","""MGSDKIHHHHHHENLYFQGMRDHVEIGIGR…",""" """,1,0,20,1
"""2QR6""","""A""","""MGSDKIHHHHHHENLYFQGMRDHVEIGIGR…",""" """,2,0,6,1
"""2QR6""","""A""","""MGSDKIHHHHHHENLYFQGMRDHVEIGIGR…",""" """,3,0,8,1
"""2QR6""","""A""","""MGSDKIHHHHHHENLYFQGMRDHVEIGIGR…",""" """,4,0,13,1
"""2QR6""","""A""","""MGSDKIHHHHHHENLYFQGMRDHVEIGIGR…",""" """,5,0,15,1
…,…,…,…,…,…,…,…
"""3PIU""","""A""","""MRMLSRNATFNSHGQDSSYFLGWQEYEKNP…","""H""",431,5,9,1
"""3PIU""","""A""","""MRMLSRNATFNSHGQDSSYFLGWQEYEKNP…","""H""",432,5,19,1
"""3PIU""","""A""","""MRMLSRNATFNSHGQDSSYFLGWQEYEKNP…",""".""",433,1,19,1


### Write to Parquet

In [18]:
output_train.collect().write_parquet('tokenized_train.parquet')
output_test.collect().write_parquet('tokenized_test.parquet')

#### Replacing Labels

In [19]:
output_train = output_train.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_train.collect().write_parquet('tokenized_train_c9.parquet')

output_test = output_test.with_columns(
                  label=pl.when(pl.col('label')==0)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_test.collect().write_parquet('tokenized_test_c9.parquet')


In [20]:
output_train = output_train.with_columns(
                  label=pl.when(pl.col('label')==1)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_train.collect().write_parquet('tokenized_train_c8.parquet')

output_test = output_test.with_columns(
                  label=pl.when(pl.col('label')==1)
                                      .then(pl.lit(-100))
                                      .otherwise(pl.col('label')))
output_test.collect().write_parquet('tokenized_test_c8.parquet')
